In [1]:
import jax 
import jax.numpy as jnp
import numpy as np
# Device cpu
#jax.config.update('jax_platform_name', 'cpu')

from probjax.nn.attention import dense_dot_product_attention, memory_efficient_dot_product_attention, MultiHeadAttention
from probjax.utils.bmutil import benchmark
import haiku as hk
from functools import partial

from jax.experimental import host_callback

/root/miniconda3/envs/probjax/lib/python3.10/site-packages/jax/_src/api_util.py:174: SyntaxWarning: Jitted function has static_argnums=(3, 4), but only accepts 4 positional arguments. This warning will be replaced by an error after 2022-08-20 at the earliest.
  warnings.warn(f"Jitted function has {argnums_name}={argnums}, "


In [8]:
x = jnp.ones((5,1000,4, 5))
mask = jax.random.bernoulli(jax.random.PRNGKey(0), 0.5, (5,1000, 1000))


dense_dot_product_attention(x, x, x, mask=mask).shape

(5, 1000, 20)

In [10]:
memory_efficient_dot_product_attention(x, x, x, mask=mask).shape

(5, 1000, 20)

In [11]:
def f_host(x):
    return jnp.sum(x + 1)

def f(x):
  result_shape = jax.ShapeDtypeStruct((), jnp.int32)
  return jax.pure_callback(f_host, result_shape, x)

f(jnp.ones((10,), dtype=jnp.int32))

Array(20, dtype=int32)

In [4]:
jax.devices()

[cuda(id=0)]

In [9]:
def f(x, mask=None, attention_method="chunked", **kwargs):
    m = MultiHeadAttention(num_heads=2, key_size=5,attention_method=attention_method, attention_kwargs=kwargs, w_init=hk.initializers.RandomNormal(1e-2))    
    return m(x,x,x, mask=mask)

init_fn, apply_fn = hk.without_apply_rng(hk.transform(f))

In [10]:
mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.1, shape=(1000, 1000)).astype(jnp.bool_)

In [11]:
x = jax.random.normal(jax.random.PRNGKey(0), (1000, 100))

In [12]:
params = init_fn(jax.random.PRNGKey(42), x)

In [13]:
mask_f = lambda : np.array(mask)

In [14]:
np.where(mask)

(array([  0,   0,   0, ..., 999, 999, 999]),
 array([ 27,  31,  42, ..., 981, 992, 995]))

In [19]:
apply_fn(params, x, mask=mask, attention_method="chunked")

TypeError: dynamic_slice slice_sizes must have the same length as start_indices, got start_indices length 3 and slice_sizes (1000, 1000).

In [33]:
%%timeit
apply_fn(params, x, mask=mask_f, attention_method="sparse").shape

2.47 ms ± 91.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [34]:
%%timeit
apply_fn(params, x, mask=mask, attention_method="dense").shape

2.73 ms ± 188 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [47]:
@jax.jit
def loss_fn(x):
    #mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.1, shape=(x.shape[0], x.shape[0]))
    return apply_fn(params, x, mask=mask, attention_method="dense").sum()

In [48]:
@jax.jit
def loss_fn2(x):
    #mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.1, shape=(x.shape[0], x.shape[0]))
    return apply_fn(params, x, mask=mask_f, attention_method="sparse").sum()

In [49]:
benchmark(loss_fn, x)

GPU Utilization: 92% +/- 13%, GPU Memory Utilization: 86% +/- 13% 
CPU Utilization: 7% +/- 3%
Memory Utilization: 2% +/- 0%
Average time taken: 212.64 us


Array(-0.137, dtype=float32)

In [50]:
benchmark(loss_fn2, x)

GPU Utilization: 72% +/- 16%, GPU Memory Utilization: 8% +/- 3% 
CPU Utilization: 7% +/- 3%
Memory Utilization: 2% +/- 0%
Average time taken: 205.44 us


Array(-0.137, dtype=float32)

In [3]:


class HashableArray():  
    
    def __init__(self, val):
        self.dtype = val.dtype
        self.val = val
    
    def __hash__(self):
        return hash(self.val.tobytes())
    
    def __jax_array__(self):
        return self.val

In [4]:
import numpy as np

In [5]:


@partial(jax.jit, static_argnums=(4,5))
def efficient_masked_dot_product_attention(
    query_heads,  # [...,T', H, K]
    key_heads,  # [...,T', H, K]
    value_heads,  # [T, H, V]
    mask=None,  # [T', T]
    size: int = None,
    return_attention_weights: bool = False,
):
    
    assert mask is not None, "Sparse attention requires a (at best sparse) mask"
    assert mask.ndim == 2, "Mask must be 2D"
    assert size is not None, "Sparse attention requires a size i.e. the number of True values in the mask aka the number of edges"
    
    *leading_dims, sequence_length, _, dim = query_heads.shape
    indices1, indices2 = jnp.where(mask, size=size)
    query_heads = jnp.take(
        query_heads, indices1, axis=-3, indices_are_sorted=True
    )  # [..., E, H, K] Where E is the number of edges
    key_heads = jnp.take(key_heads, indices2, axis=-3, indices_are_sorted=True)  # [..., E, H, K]
    value_heads = jnp.take(value_heads, indices2, axis=-3, indices_are_sorted=True)  # [..., E, H, V] 

    

    # Attention logits
    attention_logits = jnp.einsum(
        "...ehd,...ehd->...eh", query_heads, key_heads
    ) / jnp.sqrt(dim).astype(key_heads.dtype)
    attention_logits = attention_logits - jnp.max(
        attention_logits, axis=-2, keepdims=True
    )
    attention_weight = jnp.exp(attention_logits)
    attention_normalizer = jax.ops.segment_sum(
        attention_weight,
        indices1,
        num_segments=sequence_length,
        indices_are_sorted=True,
    )
    attention_normalizer = jnp.take(attention_normalizer, indices1, axis=-2)
    attention_weight = attention_weight / attention_normalizer  # [..., eh]

    # Attention weighted values
    attn = attention_weight[..., None] * value_heads
    attn = jax.ops.segment_sum(
        attn, indices1, num_segments=sequence_length, indices_are_sorted=True
    )
    attn = jnp.reshape(attn, (*leading_dims, sequence_length, -1))  # [T', H*V]

    if return_attention_weights:
        return attn, attention_weight
    else:
        return attn, None

In [93]:
def mask_fn():
    return np.array(mask)

In [94]:
def f(x, mask_fn):
    mask = mask_fn()
    return efficient_masked_dot_product_attention(x, x, x, mask=mask[0], size=int(mask[0].sum()))

In [98]:
def g(x, mask):
    return efficient_masked_dot_product_attention(x, x, x, mask=mask, size=int(mask[0].sum()))

In [99]:
mask = np.array(mask)

In [97]:
%%timeit
jax.jit(f, static_argnums=(1,))(q, mask_fn)

13.5 ms ± 25.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [105]:
%%timeit
jax.jit(g, static_argnums=(1,))(q, HashableArray(mask))

TypeError: 'HashableArray' object is not subscriptable

In [69]:
%%timeit
jnp.where(mask[0])

33 ms ± 199 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:

mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.01, shape=(1, 6000, 6000)).astype(bool)

q = jax.random.normal(jax.random.PRNGKey(0), shape=(6000,8, 10))
k = jax.random.normal(jax.random.PRNGKey(0), shape=(6000,8, 10))
v = jax.random.normal(jax.random.PRNGKey(0), shape=(6000,8, 10))

In [12]:
%%timeit
dense_dot_product_attention(q, k, v, mask=mask)[0]

42.7 ms ± 6.9 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [46]:
%%timeit
efficient_masked_dot_product_attention(q, k, v, mask=mask[0], size=mask[0].sum())

ValueError: Non-hashable static arguments are not supported. An error occurred during a call to 'efficient_masked_dot_product_attention' while trying to hash an object of type <class 'jaxlib.xla_extension.ArrayImpl'>, 359428. The error was:
TypeError: unhashable type: 'ArrayImpl'


In [ ]:
def f():
    return 

In [15]:
from probjax.utils.bmutil import benchmark

In [16]:
import math

In [17]:
2048 // math.gcd(2048, 2048)

1

In [18]:
_ = benchmark(efficient_masked_dot_product_attention, q, k, v, HashableArray(mask[0]), size=int(mask.sum()))

GPU Utilization: 98% +/- 8%, GPU Memory Utilization: 26% +/- 2% 
CPU Utilization: 1% +/- 2%
Memory Utilization: 2% +/- 0%
Average time taken: 18.46 ms


In [20]:
_ = benchmark(efficient_masked_dot_product_attention, q, k, v, mask[0], size=int(mask.sum()))

GPU Utilization: 96% +/- 15%, GPU Memory Utilization: 26% +/- 5% 
CPU Utilization: 0% +/- 2%
Memory Utilization: 2% +/- 0%
Average time taken: 18.34 ms


In [19]:
_ = benchmark(dense_dot_product_attention, q, k, v, mask=mask)

GPU Utilization: 96% +/- 16%, GPU Memory Utilization: 95% +/- 18% 
CPU Utilization: 0% +/- 1%
Memory Utilization: 2% +/- 0%
Average time taken: 34.50 ms


In [32]:
_ = benchmark(memory_efficient_dot_product_attention, q, k, v, mask=mask, query_chunk_size=512)

GPU Utilization: 96% +/- 15%, GPU Memory Utilization: 65% +/- 11% 
CPU Utilization: 7% +/- 2%
Memory Utilization: 2% +/- 0%
Average time taken: 59.61 ms


Average time: 0.3580264401435852Memory Utilization: 13% +/- 10% 


In [8]:
def gpu_in_use_by_this_process(gpu_handle: "GPUHandle", pid: int) -> bool:
    if psutil is None:
        return False

    try:
        base_process = psutil.Process(pid=pid)
    except psutil.NoSuchProcess:
        # do not report any gpu metrics if the base process can't be found
        return False

    our_processes = base_process.children(recursive=True)
    our_processes.append(base_process)

    our_pids = {process.pid for process in our_processes}

    compute_pids = {
        process.pid
        for process in pynvml.nvmlDeviceGetComputeRunningProcesses(gpu_handle)  # type: ignore
    }
    graphics_pids = {
        process.pid
        for process in pynvml.nvmlDeviceGetGraphicsRunningProcesses(gpu_handle)  # type: ignore
    }

    pids_using_device = compute_pids | graphics_pids

    return len(pids_using_device & our_pids) > 0

In [108]:
dense_dot_product_attention(q, k, v, mask=mask[0][None], key_size=k.shape[-1])[0]

Array([[ 0.041, -0.327, -0.157, ..., -0.048,  0.316, -0.296],
       [-0.061, -0.402, -0.534, ..., -0.276,  0.884, -0.136],
       [ 0.789,  0.283,  0.69 , ..., -0.379,  0.06 , -0.048],
       ...,
       [-0.941, -0.359,  0.334, ...,  0.087, -0.209,  0.451],
       [-0.058, -0.858, -1.623, ...,  0.896, -0.705, -0.579],
       [-0.307, -0.163, -0.135, ..., -0.244,  0.485,  0.001]],      dtype=float32)

In [25]:
@jax.jit
def f1():
    return dense_dot_product_attention(q, k, v, mask=mask[0][None])[0]


In [26]:
@jax.jit
def f2():
    return memory_efficient_dot_product_attention(q, k, v, mask=mask[0][None])

In [40]:
def f3():
    return efficient_masked_dot_product_attention(q, k, v, mask=mask[0][None])

In [28]:
out1 = f1()

In [29]:
out2 = f2()

In [41]:
out3 = f3()

AssertionError: Sparse attention requires a size i.e. the number of True values in the mask aka the number of edges

In [36]:
jnp.allclose(out1, out2, atol=1e-2)

Array(True, dtype=bool)

In [38]:
out1

Array([[ 0.351,  0.096,  0.329, ...,  0.063, -0.481,  0.625],
       [-0.63 ,  0.242,  0.837, ...,  0.139,  0.021, -0.155],
       [-0.188, -0.119, -0.13 , ..., -0.43 , -0.11 ,  0.566],
       ...,
       [ 0.402, -0.415, -0.264, ...,  0.267, -0.002, -0.724],
       [ 0.02 , -0.006,  0.316, ...,  0.112,  0.245, -0.069],
       [-0.054, -0.051,  0.301, ..., -0.429,  0.153,  0.183]],      dtype=float32)

In [39]:
out2

Array([[ 0.351,  0.097,  0.33 , ...,  0.063, -0.482,  0.626],
       [-0.63 ,  0.242,  0.838, ...,  0.139,  0.021, -0.155],
       [-0.188, -0.119, -0.13 , ..., -0.43 , -0.11 ,  0.566],
       ...,
       [ 0.403, -0.416, -0.265, ...,  0.267, -0.002, -0.724],
       [ 0.02 , -0.006,  0.316, ...,  0.112,  0.245, -0.069],
       [-0.055, -0.052,  0.302, ..., -0.43 ,  0.154,  0.183]],      dtype=float32)

In [21]:
%%timeit
f1()

38.3 µs ± 1.4 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [263]:
%%timeit
f2()

84.3 ms ± 25.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [23]:
with jax.profiler.trace("/tmp/tensorboard"):
  # Run the operations to be profiled
  y = f1()
  y.block_until_ready()

2024-02-23 17:21:10.870816: E external/xla/xla/python/profiler/internal/python_hooks.cc:398] Can't import tensorflow.python.profiler.trace
2024-02-23 17:21:10.883745: E external/xla/xla/backends/profiler/gpu/cupti_error_manager.cc:135] cuptiGetTimestamp: ignored due to a previous error.
2024-02-23 17:21:10.883776: E external/xla/xla/backends/profiler/gpu/cupti_error_manager.cc:186] cuptiSubscribe: ignored due to a previous error.
2024-02-23 17:21:10.883780: E external/xla/xla/backends/profiler/gpu/cupti_error_manager.cc:459] cuptiGetResultString: ignored due to a previous error.
2024-02-23 17:21:10.883783: E external/xla/xla/backends/profiler/gpu/cupti_tracer.cc:1935] function cupti_interface_->Subscribe( &subscriber_, (CUpti_CallbackFunc)ApiCallback, this)failed with error 
2024-02-23 17:21:10.890479: E external/xla/xla/python/profiler/internal/python_hooks.cc:398] Can't import tensorflow.python.profiler.trace
2024-02-23 17:21:10.892929: E external/xla/xla/backends/profiler/gpu/cupti_

In [23]:
with jax.profiler.trace("/tmp/tensorboard"):
  # Run the operations to be profiled
  y = f2()
  y.block_until_ready()